In [2]:
import pandas as pd
import re

input_log_path = "/Users/fredikamionka/PycharmProjects/Projects_for_CV/Files_Project_for_CV/logs.txt"
output_parquet_path = "/Users/fredikamionka/PycharmProjects/Projects_for_CV/Files_Project_for_CV/logs_clean.parquet"

log_pattern = re.compile(
    r'(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) '
    r'(?P<level>INFO|WARN|ERROR) '
    r'(?P<message>.*) '
    r'user_id=(?P<user_id>\d+) '
    r'ip=(?P<ip>\d+\.\d+\.\d+\.\d+)'
)

parsed_records = []

with open(input_log_path, "r") as log_file:

    for line in log_file:

        line = line.strip()
        match = log_pattern.match(line)

        if match:

            parsed_record = match.groupdict()
            parsed_record["status"] = "ok"

        else:

            parsed_record = {
                "timestamp": None,
                "level": None,
                "message": None,
                "user_id": None,
                "ip": None,
                "status": "error",
                "raw_line": line
            }

        parsed_records.append(parsed_record)

parsed_logs_df = pd.DataFrame(parsed_records)

parsed_logs_df["timestamp"] = pd.to_datetime(parsed_logs_df["timestamp"], errors="coerce")
parsed_logs_df["user_id"] = pd.to_numeric(parsed_logs_df["user_id"], errors="coerce")

parsed_logs_df.to_parquet(output_parquet_path)

print(parsed_logs_df.head())
print(parsed_logs_df["status"].value_counts())

            timestamp  level           message  user_id             ip status  \
0 2026-04-09 08:00:00   WARN       User logout     79.0   192.168.1.22     ok   
1 2026-04-09 08:00:04   INFO        User login     53.0  192.168.1.203     ok   
2 2026-04-09 08:00:08  ERROR  Password changed     32.0   192.168.1.61     ok   
3 2026-04-09 08:00:09  ERROR       User logout    117.0  192.168.1.188     ok   
4 2026-04-09 08:00:16  ERROR        User login    105.0   192.168.1.98     ok   

  raw_line  
0      NaN  
1      NaN  
2      NaN  
3      NaN  
4      NaN  
status
error    882
ok       114
Name: count, dtype: int64


In [3]:
print(parsed_logs_df[parsed_logs_df["status"] == "error"]["raw_line"].head(20))

56    SYSTEM GLITCH ### 2026-04-09 08:00:57 WARN Set...
57    ip=192.168.1.33 2026-04-09 08:03:52 INFO Passw...
58    ip=192.168.1.14 2026-04-09 08:01:58 WARN Setti...
59    ip=192.168.1.40 2026-04-09 08:03:00 ERROR User...
60    ip=192.168.1.211 2026-04-09 08:02:02 WARN User...
61    ip=192.168.1.151 2026-04-09 08:01:02 WARN User...
62    ip=192.168.1.225 2026-04-09 08:03:09 INFO File...
63    ip=192.168.1.242 2026-04-09 08:01:04 INFO Pass...
64    ip=192.168.1.11 2026-04-09 08:03:15 INFO User ...
65    ip=192.168.1.11 2026-04-09 08:02:12 INFO Setti...
66    ip=192.168.1.178 2026-04-09 08:04:28 WARN User...
67    ip=192.168.1.182 2026-04-09 08:01:08 WARN Pass...
68    ip=192.168.1.8 2026-04-09 08:02:18 INFO User l...
69    ip=192.168.1.224 2026-04-09 08:04:40 WARN File...
70    ip=192.168.1.49 2026-04-09 08:02:22 ERROR Sett...
71    ip=192.168.1.4 2026-04-09 08:02:24 INFO User l...
72    ip=192.168.1.207 2026-04-09 08:01:13 ERROR Fil...
73    ip=192.168.1.110 2026-04-09 08:02:28 WARN 